# Lab 2 — Designing an OCI-Based AIOps Reference Architecture
### AIOps Professional 2026 | Oracle Cloud Infrastructure

---

## What this lab does

Session 2 introduced the three layers of every AIOps platform:

| Layer | What it does |
|---|---|
| **Data Layer** | Collects logs, metrics, events, traces from every source |
| **ML Layer** | Detects anomalies, correlates events, predicts failures |
| **Automation Layer** | Remediates, notifies, feeds results back into the pipeline |

In this lab you will:
- **Visualise the full OCI AIOps reference architecture** as a live component diagram
- **Simulate real data flowing** through each layer — from raw OCI telemetry to automated action
- **Trace a complete incident** through the pipeline end-to-end
- **Design your own architecture** by choosing OCI services for each layer and evaluating your choices
- **Produce a written architecture decision record** you can take back to your team

> Run each cell with `Shift + Enter`. No OCI credentials required — all data is simulated.

---

## Section 1 — Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')
np.random.seed(7)

plt.rcParams.update({
    'figure.facecolor': '#F8F9FA',
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# ── colour palette (matches OCI branding) ─────────────────────────────────────
OCI_RED    = '#C74634'
OCI_NAVY   = '#1F3864'
OCI_BLUE   = '#2E75B6'
OCI_GREEN  = '#375623'
OCI_AMBER  = '#FF8C00'
OCI_PURPLE = '#7030A0'
OCI_TEAL   = '#006E6E'
OCI_LGRAY  = '#F2F2F2'

print('✅  Setup complete.')

---
## Section 2 — The OCI AIOps Reference Architecture

Run the cell below to render the full OCI AIOps reference architecture diagram.

Every box is a real OCI service. Every arrow is a real data flow.
Study the diagram before moving to Section 3.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 13))
ax.set_xlim(0, 18)
ax.set_ylim(0, 13)
ax.axis('off')
fig.patch.set_facecolor('#FAFBFC')

def box(ax, x, y, w, h, label, sublabel='', color=OCI_BLUE, textcolor='white', fontsize=9, radius=0.3):
    rect = FancyBboxPatch((x, y), w, h,
                          boxstyle=f'round,pad=0.05,rounding_size={radius}',
                          facecolor=color, edgecolor='white', linewidth=1.5, zorder=3)
    ax.add_patch(rect)
    cy = y + h / 2 + (0.12 if sublabel else 0)
    ax.text(x + w/2, cy, label, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=textcolor, zorder=4, wrap=True)
    if sublabel:
        ax.text(x + w/2, y + h/2 - 0.22, sublabel, ha='center', va='center',
                fontsize=7.5, color=textcolor, zorder=4, alpha=0.88)

def arrow(ax, x1, y1, x2, y2, label='', color='#555555'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.6), zorder=5)
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx, my + 0.13, label, ha='center', fontsize=7, color=color, style='italic', zorder=6)

def layer_bg(ax, x, y, w, h, label, color, alpha=0.07):
    rect = FancyBboxPatch((x, y), w, h,
                          boxstyle='round,pad=0.1,rounding_size=0.4',
                          facecolor=color, edgecolor=color, linewidth=2,
                          alpha=alpha, zorder=1)
    ax.add_patch(rect)
    ax.text(x + 0.18, y + h - 0.28, label, fontsize=8.5, fontweight='bold',
            color=color, alpha=0.85, zorder=2)

# ── LAYER BACKGROUNDS ─────────────────────────────────────────────────────────
layer_bg(ax, 0.3, 10.2, 17.4, 2.5,  '① DATA SOURCES',             OCI_NAVY,   0.08)
layer_bg(ax, 0.3,  7.4, 17.4, 2.5,  '② DATA INGESTION & STORAGE', OCI_BLUE,   0.07)
layer_bg(ax, 0.3,  4.3, 17.4, 2.8,  '③ ML & ANALYTICS',           OCI_PURPLE, 0.07)
layer_bg(ax, 0.3,  1.3, 17.4, 2.7,  '④ AUTOMATION & FEEDBACK',    OCI_GREEN,  0.07)

# ── LAYER 1: DATA SOURCES ─────────────────────────────────────────────────────
box(ax,  0.6, 10.6, 2.2, 1.6, 'OKE / Kubernetes',  'Pods, Nodes,\nControllers',  OCI_NAVY)
box(ax,  3.1, 10.6, 2.2, 1.6, 'OCI Compute',       'VMs, BMs,\nAutoscaling',     OCI_NAVY)
box(ax,  5.6, 10.6, 2.2, 1.6, 'Autonomous DB',     'Queries,\nWait Events',       OCI_NAVY)
box(ax,  8.1, 10.6, 2.2, 1.6, 'Load Balancer',     'Requests,\nLatency, Errors',  OCI_NAVY)
box(ax, 10.6, 10.6, 2.2, 1.6, 'OCI Functions',     'Invocations,\nErrors',        OCI_NAVY)
box(ax, 13.1, 10.6, 2.2, 1.6, 'API Gateway',       'Calls, Status\nCodes',        OCI_NAVY)
box(ax, 15.6, 10.6, 1.9, 1.6, 'Custom Apps',       'Business\nMetrics',           OCI_NAVY)

# ── LAYER 2: INGESTION & STORAGE ─────────────────────────────────────────────
box(ax,  0.6, 7.8, 2.8, 1.6, 'OCI Logging',        'Structured &\nUnstructured',  OCI_BLUE)
box(ax,  3.7, 7.8, 2.8, 1.6, 'OCI Monitoring',     'Metrics,\nAlarms, Queries',   OCI_BLUE)
box(ax,  6.8, 7.8, 2.8, 1.6, 'Service Connector\nHub', 'Route streams\nto targets', OCI_BLUE)
box(ax,  9.9, 7.8, 2.8, 1.6, 'OCI Streaming',      'Kafka-compatible\nEventBus',  OCI_TEAL)
box(ax, 13.0, 7.8, 2.8, 1.6, 'Logging Analytics',  'Search, Query,\nDashboards',  OCI_BLUE)
box(ax, 16.1, 7.8, 1.5, 1.6, 'Object\nStorage',    'Archive &\nML training',      OCI_BLUE)

# ── LAYER 3: ML & ANALYTICS ──────────────────────────────────────────────────
box(ax,  0.6, 4.7, 3.0, 1.7, 'OCI Data Science',   'Model Training,\nAnomaly Detection', OCI_PURPLE)
box(ax,  4.0, 4.7, 3.0, 1.7, 'OCI AI Services',    'Anomaly Detection\nService (ADS)',    OCI_PURPLE)
box(ax,  7.4, 4.7, 3.0, 1.7, 'OCI GenAI Service',  'LLMs, RAG,\nkubectl-ai',              OCI_PURPLE)
box(ax, 10.8, 4.7, 3.0, 1.7, 'Oracle 23ai',        'Vector Store,\nEmbeddings',           OCI_PURPLE)
box(ax, 14.2, 4.7, 3.3, 1.7, 'Correlation Engine', 'Event Grouping,\nRCA, Topology',      OCI_AMBER,  textcolor='white')

# ── LAYER 4: AUTOMATION & FEEDBACK ───────────────────────────────────────────
box(ax,  0.6, 1.7, 2.8, 1.5, 'OCI DevOps',         'Pipelines,\nSelf-Healing',    OCI_GREEN)
box(ax,  3.7, 1.7, 2.8, 1.5, 'OCI Events &\nAlarms', 'Notifications,\nPagerDuty', OCI_GREEN)
box(ax,  6.8, 1.7, 2.8, 1.5, 'OCI Functions',      'Serverless\nRemediation',     OCI_GREEN)
box(ax,  9.9, 1.7, 2.8, 1.5, 'ITSM / ServiceNow',  'Ticket Creation,\nEscalation', OCI_GREEN)
box(ax, 13.0, 1.7, 2.8, 1.5, 'OCI Notifications',  'Email, Slack,\nPagerDuty',    OCI_GREEN)
box(ax, 16.1, 1.7, 1.5, 1.5, 'Feedback\nLoop',     'Model\nRetraining',            OCI_RED)

# ── ARROWS: DATA SOURCES → INGESTION ─────────────────────────────────────────
for sx in [1.7, 4.2, 6.7, 9.2, 11.7, 14.2, 16.55]:
    arrow(ax, sx, 10.6, sx, 9.4, color='#999999')

# ── ARROWS: INGESTION → ML ────────────────────────────────────────────────────
arrow(ax, 2.0,  7.8, 2.1,  6.4,  color=OCI_BLUE)
arrow(ax, 5.1,  7.8, 5.5,  6.4,  color=OCI_BLUE)
arrow(ax, 8.2,  7.8, 8.9,  6.4,  color=OCI_TEAL)
arrow(ax, 11.3, 7.8, 12.3, 6.4,  color=OCI_BLUE)
arrow(ax, 14.4, 7.8, 15.8, 6.4,  color=OCI_BLUE)

# ── ARROWS: ML → AUTOMATION ──────────────────────────────────────────────────
arrow(ax, 2.1,  4.7, 2.0,  3.2,  color=OCI_PURPLE)
arrow(ax, 5.5,  4.7, 5.1,  3.2,  color=OCI_PURPLE)
arrow(ax, 8.9,  4.7, 8.2,  3.2,  color=OCI_PURPLE)
arrow(ax, 12.3, 4.7, 11.3, 3.2,  color=OCI_PURPLE)
arrow(ax, 15.8, 4.7, 14.4, 3.2,  color=OCI_AMBER)

# ── FEEDBACK LOOP ARROW ───────────────────────────────────────────────────────
ax.annotate('', xy=(16.85, 7.8), xytext=(16.85, 3.2),
            arrowprops=dict(arrowstyle='->', color=OCI_RED, lw=2.0,
                            connectionstyle='arc3,rad=0.0'), zorder=5)
ax.text(17.2, 5.5, 'Feedback\nLoop', ha='center', fontsize=8, color=OCI_RED,
        fontweight='bold', rotation=90)

# ── TITLE ─────────────────────────────────────────────────────────────────────
ax.text(9.0, 12.75, 'OCI AIOps Reference Architecture',
        ha='center', fontsize=16, fontweight='bold', color=OCI_NAVY)
ax.text(9.0, 12.45, 'Four-Layer Platform: Data Sources → Ingestion & Storage → ML & Analytics → Automation & Feedback',
        ha='center', fontsize=9, color='#555555')

plt.tight_layout()
plt.savefig('lab2_oci_architecture.png', dpi=130, bbox_inches='tight')
plt.show()
print('\nFigure saved: lab2_oci_architecture.png')

### 🔍 Study the diagram — answer these before moving on

1. A Kubernetes pod crashes at 2am. **Which box in Layer 1 generates the signal?**
2. That signal travels to Layer 2. **Which two OCI services would receive it — logging AND metrics?**
3. Before ML can detect an anomaly, data must be routed. **Which Layer 2 service acts as the central router?**
4. The ML layer produces a diagnosis. **Which Layer 4 service triggers the fix without human intervention?**
5. After the fix, the model needs to learn from what happened. **Which box closes that loop?**

> Answers will become obvious after Section 3.

---
## Section 3 — Simulating Data Flow Through the Architecture

Watch a real OCI operational event travel through all four layers.

**Scenario:** An OKE pod serving a payment API starts failing. We trace the data from the moment the first metric deviates until automated remediation completes.

In [ ]:
from datetime import datetime, timedelta
import time

# ── Simulate the event timeline ───────────────────────────────────────────────
t0 = datetime(2026, 4, 10, 14, 0, 0)   # incident start time

pipeline_events = [
    # (offset_sec, layer, service,              event_description,                           data_payload)
    (  0, 1, 'OKE / Kubernetes',        'Pod payment-api-7f9b restartCount increments',
             {'pod': 'payment-api-7f9b', 'restartCount': 3, 'reason': 'OOMKilled', 'memory_mb': 512}),

    (  2, 1, 'OCI Compute',             'Node cpu_utilisation crosses 85%',
             {'node': 'worker-node-02', 'cpu_pct': 87.3, 'threshold': 80}),

    (  4, 2, 'OCI Logging',             'Container log ingested: OOMKilled event',
             {'log_level': 'ERROR', 'source': 'payment-api-7f9b', 'msg': 'OOMKilled — memory limit exceeded'}),

    (  5, 2, 'OCI Monitoring',          'Alarm FIRING: pod restart rate > 2 in 5min',
             {'metric': 'kubernetes.pods.restart_rate', 'value': 3, 'alarm': 'pod-restart-alarm', 'state': 'FIRING'}),

    (  6, 2, 'Service Connector Hub',   'Routing log stream → OCI Streaming topic: k8s-events',
             {'source': 'OCI Logging', 'target': 'OCI Streaming', 'topic': 'k8s-events', 'records': 1}),

    (  8, 2, 'OCI Streaming',           'Event published to stream for ML consumption',
             {'stream': 'k8s-events', 'partition': 0, 'offset': 10043, 'consumer_group': 'aiops-ml-consumer'}),

    ( 12, 3, 'OCI AI Services',         'Anomaly Detection model scores: ANOMALY (score 0.94)',
             {'model': 'pod-memory-anomaly-v2', 'signal': 'memory_usage', 'anomaly_score': 0.94, 'threshold': 0.75}),

    ( 14, 3, 'OCI GenAI Service',       'kubectl-ai diagnosis: Root cause = memory limit too low for current load',
             {'prompt': 'Why did payment-api-7f9b get OOMKilled?', 'diagnosis': 'Memory limit 512Mi too low — recommend 768Mi', 'confidence': 0.91}),

    ( 16, 3, 'Correlation Engine',      'Correlated: 3 alerts grouped → 1 incident (root cause: payment-api-7f9b)',
             {'alerts_grouped': 3, 'incident_id': 'INC-2041', 'root_cause': 'payment-api-7f9b OOMKilled', 'affected': ['payment-api-7f9b', 'worker-node-02', 'Load Balancer']}),

    ( 20, 4, 'OCI Functions',           'Serverless function triggered: patch_deployment_memory()',
             {'function': 'patch-k8s-memory', 'action': 'kubectl patch deployment payment-api --patch memory=768Mi', 'invocation_id': 'fn-8821'}),

    ( 23, 4, 'OCI DevOps',              'Self-healing pipeline triggered: rolling restart with new memory limit',
             {'pipeline': 'payment-api-self-heal', 'stage': 'Rolling Restart', 'new_memory': '768Mi', 'status': 'RUNNING'}),

    ( 28, 4, 'ITSM / ServiceNow',       'Incident INC-2041 auto-created and auto-resolved',
             {'incident': 'INC-2041', 'state': 'RESOLVED', 'resolution': 'Memory limit patched automatically', 'MTTR_sec': 28}),

    ( 30, 4, 'Feedback Loop',           'Outcome logged → anomaly model updated with new normal baseline',
             {'event': 'model_retrain_triggered', 'model': 'pod-memory-anomaly-v2', 'new_samples': 47, 'scheduled_retrain': '02:00 UTC'}),
]

# ── Print the pipeline trace ───────────────────────────────────────────────────
layer_labels = {1: 'DATA SOURCES', 2: 'INGESTION & STORAGE', 3: 'ML & ANALYTICS', 4: 'AUTOMATION & FEEDBACK'}
layer_colors = {1: '\033[94m', 2: '\033[96m', 3: '\033[95m', 4: '\033[92m'}
RESET = '\033[0m'
BOLD  = '\033[1m'

current_layer = 0
print(f'{BOLD}INCIDENT PIPELINE TRACE  —  INC-2041{RESET}')
print(f'Start time: {t0.strftime("%d %b %Y %H:%M:%S")}\n')

for offset, layer, service, description, payload in pipeline_events:
    ts = (t0 + timedelta(seconds=offset)).strftime('%H:%M:%S')
    if layer != current_layer:
        current_layer = layer
        lc = layer_colors[layer]
        print(f'\n  {lc}{BOLD}▌ Layer {layer}: {layer_labels[layer]}{RESET}')
        print(f'  {"─" * 60}')
    print(f'  {BOLD}[+{offset:>3}s]  {ts}{RESET}  {service}')
    print(f'          {description}')
    # Show key payload fields
    key_fields = list(payload.items())[:3]
    field_str  = '  |  '.join([f'{k}: {v}' for k, v in key_fields])
    print(f'          → {field_str}')

print(f'\n  {BOLD}Total pipeline time: 30 seconds  |  Human involvement: 0{RESET}')

### Visualise the pipeline timing

In [ ]:
DARK = '#404040'  # ensure available in this cell
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#FAFBFC')

layer_colour_map = {1: OCI_NAVY, 2: OCI_BLUE, 3: OCI_PURPLE, 4: OCI_GREEN}
layer_name_map   = {1: 'Data Sources', 2: 'Ingestion & Storage', 3: 'ML & Analytics', 4: 'Automation & Feedback'}

y_positions = {svc: idx for idx, (_, _, svc, _, _) in enumerate(reversed(pipeline_events))}

for offset, layer, service, description, payload in pipeline_events:
    y = y_positions[service]
    color = layer_colour_map[layer]
    ax.barh(y, 2.5, left=offset, height=0.6, color=color, alpha=0.85, edgecolor='white', linewidth=0.8)
    ax.text(offset + 2.7, y, service, va='center', fontsize=8.5, color=DARK)
    ax.text(offset + 1.25, y, f'+{offset}s', va='center', ha='center', fontsize=7.5, color='white', fontweight='bold')

# Layer legend
handles = [mpatches.Patch(color=layer_colour_map[l], label=f'Layer {l}: {layer_name_map[l]}') for l in range(1,5)]
ax.legend(handles=handles, loc='lower right', fontsize=9, framealpha=0.9)

ax.set_xlabel('Seconds from incident start', fontsize=11)
ax.set_title('AIOps Pipeline — Time from Raw Event to Automated Resolution\nINC-2041: OKE Pod OOMKilled → Auto-Remediated in 30 seconds',
             fontsize=12, fontweight='bold')
ax.set_yticks([])
ax.set_xlim(-1, 40)
ax.axvline(30, color=OCI_RED, linewidth=2, linestyle='--', alpha=0.7)
ax.text(30.3, len(pipeline_events)-0.5, 'Resolved\n+30s', color=OCI_RED, fontsize=9, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('lab2_pipeline_timing.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figure saved: lab2_pipeline_timing.png')

---
## Section 4 — The Feedback Loop

The feedback loop is what separates AIOps from simple automation.
Every resolved incident makes the system smarter.

Run the cell to see how the anomaly model improves across 10 incidents.

In [ ]:
# ── Simulate model improvement over repeated incidents ────────────────────────
incidents = list(range(1, 11))

# Without feedback loop — static threshold, MTTR stays roughly the same
mttr_no_feedback  = [93, 88, 95, 91, 87, 90, 89, 92, 88, 91]
false_pos_no_fb   = [14, 13, 15, 14, 13, 14, 13, 14, 13, 14]

# With feedback loop — each incident retrains model, detection improves
mttr_with_feedback = [93, 72, 55, 41, 32, 27, 22, 18, 15, 13]
false_pos_with_fb  = [14, 11,  9,  7,  5,  4,  3,  2,  2,  1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feedback Loop: Model Improvement Across Incidents', fontsize=13, fontweight='bold')

# MTTR improvement
ax1.plot(incidents, mttr_no_feedback,  'o--', color=OCI_RED,   linewidth=2, markersize=7, label='Without feedback loop')
ax1.plot(incidents, mttr_with_feedback,'o-',  color=OCI_GREEN, linewidth=2.5, markersize=7, label='With feedback loop')
ax1.fill_between(incidents, mttr_no_feedback, mttr_with_feedback, alpha=0.1, color=OCI_GREEN)
ax1.set_xlabel('Incident Number', fontsize=11)
ax1.set_ylabel('MTTR (minutes)', fontsize=11)
ax1.set_title('Mean Time to Resolve (MTTR)', fontsize=11)
ax1.legend(fontsize=10)
ax1.set_xticks(incidents)
ax1.set_ylim(0, 110)
for i, (nf, wf) in enumerate(zip(mttr_no_feedback, mttr_with_feedback)):
    if i in [0, 4, 9]:
        ax1.annotate(f'{wf}m', xy=(incidents[i], wf), xytext=(incidents[i]+0.15, wf+4),
                     fontsize=8, color=OCI_GREEN, fontweight='bold')

# False positive reduction
ax2.plot(incidents, false_pos_no_fb,  'o--', color=OCI_RED,   linewidth=2, markersize=7, label='Without feedback loop')
ax2.plot(incidents, false_pos_with_fb,'o-',  color=OCI_GREEN, linewidth=2.5, markersize=7, label='With feedback loop')
ax2.fill_between(incidents, false_pos_no_fb, false_pos_with_fb, alpha=0.1, color=OCI_GREEN)
ax2.set_xlabel('Incident Number', fontsize=11)
ax2.set_ylabel('False Positive Alerts per Incident', fontsize=11)
ax2.set_title('Alert Noise Reduction', fontsize=11)
ax2.legend(fontsize=10)
ax2.set_xticks(incidents)
ax2.set_ylim(0, 20)
ax2.annotate('93% reduction\nby incident 10', xy=(10, 1), xytext=(7.5, 8),
             arrowprops=dict(arrowstyle='->', color=OCI_GREEN),
             fontsize=9, color=OCI_GREEN, fontweight='bold')

plt.tight_layout()
plt.savefig('lab2_feedback_loop.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figure saved: lab2_feedback_loop.png')

### 🔍 What the feedback loop actually does in OCI

| Step | OCI Service | What happens |
|---|---|---|
| Incident resolved | OCI DevOps / OCI Functions | Resolution outcome written to Object Storage |
| Outcome labelled | Logging Analytics | Log labelled as `true_positive` or `false_positive` |
| Model retrained | OCI Data Science | New training job picks up labelled data from Object Storage |
| New model deployed | OCI AI Services | Updated anomaly thresholds published to detection service |
| System smarter | All layers | Lower MTTR and fewer false positives on next incident |

---
## Section 5 — Design Your Own OCI AIOps Architecture

Now you design the architecture for a real scenario.

### Your Scenario
Your organisation runs a **customer-facing e-commerce platform on OCI**:
- 6 OKE microservices (cart, payments, inventory, auth, notifications, frontend)
- 2 Autonomous Databases (transactional + analytics)
- OCI Load Balancer + API Gateway
- 50,000 transactions per hour at peak
- Current state: NOC team gets ~200 alerts/day, MTTR is 2 hours average
- Goal: reduce alerts by 80%, MTTR to under 15 minutes

### Instructions
For each layer, choose the OCI services you would use.
Edit the dictionaries below — set `True` to include a service, `False` to exclude it.
Then run the cell to evaluate and justify your architecture.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  YOUR ARCHITECTURE DECISIONS — change True/False for each service
# ─────────────────────────────────────────────────────────────────────────────

layer1_sources = {
    'OKE / Kubernetes':   True,   # Must-have — your workloads run here
    'OCI Compute':        True,   # Underlying nodes
    'Autonomous DB':      True,   # DB metrics and query events
    'Load Balancer':      True,   # Traffic and error rates
    'API Gateway':        True,   # API call metrics
    'OCI Functions':      False,  # No serverless workloads in this scenario
    'Custom App Metrics': True,   # Business KPIs (orders/sec, cart abandonment)
}

layer2_ingestion = {
    'OCI Logging':              True,   # Collect all container and infra logs
    'OCI Monitoring':           True,   # Metrics and alarms
    'OCI Logging Analytics':    True,   # Query and dashboards
    'Service Connector Hub':    True,   # Route streams between services
    'OCI Streaming':            True,   # Real-time event bus for ML
    'Object Storage':           True,   # Archive logs for ML training
    'Application Perf Monitor': False,  # APM — useful but not critical for MVP
}

layer3_ml = {
    'OCI Anomaly Detection Service': True,   # Detect metric deviations
    'OCI Data Science':              True,   # Custom models, retraining
    'OCI GenAI Service':             True,   # LLM-based diagnosis and kubectl-ai
    'Oracle 23ai Vector Store':      True,   # Store incident embeddings for RAG
    'Correlation Engine':            True,   # Group related alerts
    'OCI Language Service':          False,  # NLP on logs — useful later, not MVP
}

layer4_automation = {
    'OCI DevOps Pipelines':   True,   # Self-healing pipeline execution
    'OCI Functions':          True,   # Lightweight serverless remediation
    'OCI Events & Alarms':    True,   # Trigger automation on threshold breach
    'OCI Notifications':      True,   # Alert NOC team for unresolved incidents
    'ITSM / ServiceNow':      True,   # Auto-create + auto-close tickets
    'Feedback Loop Storage':  True,   # Write outcomes back to Object Storage
}

# ─────────────────────────────────────────────────────────────────────────────

all_layers = [
    ('Layer 1 — Data Sources',           layer1_sources),
    ('Layer 2 — Ingestion & Storage',    layer2_ingestion),
    ('Layer 3 — ML & Analytics',         layer3_ml),
    ('Layer 4 — Automation & Feedback',  layer4_automation),
]

print('YOUR OCI AIOps ARCHITECTURE')
print('=' * 55)
total_selected = 0
for layer_name, services in all_layers:
    selected   = [s for s, v in services.items() if v]
    deselected = [s for s, v in services.items() if not v]
    total_selected += len(selected)
    print(f'\n  {layer_name}')
    print(f'  {"─"*50}')
    for s in selected:    print(f'    ✅  {s}')
    for s in deselected:  print(f'    ⬜  {s}  (excluded)')
print(f'\n  Total services selected: {total_selected}')

### Evaluate your architecture

In [ ]:
# ── Scoring rules (each missing critical service reduces score) ───────────────
critical = {
    'OCI Logging':                   (layer2_ingestion, 'Cannot collect logs without this — entire pipeline fails'),
    'OCI Monitoring':                (layer2_ingestion, 'No metric alarms without this — cannot trigger any automation'),
    'Service Connector Hub':         (layer2_ingestion, 'Without routing, logs and metrics cannot reach ML layer'),
    'OCI Streaming':                 (layer2_ingestion, 'Real-time ML requires a streaming event bus'),
    'OCI Anomaly Detection Service': (layer3_ml,        'Core ML detection — without this, AIOps reduces to scripted alerting'),
    'Correlation Engine':            (layer3_ml,        'Without correlation, alert storm problem remains unsolved'),
    'OCI DevOps Pipelines':          (layer4_automation,'Self-healing requires pipeline execution capability'),
    'Feedback Loop Storage':         (layer4_automation,'Without feedback, model never improves — AIOps degrades over time'),
}

recommended = {
    'OCI GenAI Service':         (layer3_ml,        'LLM diagnosis and kubectl-ai significantly reduce manual RCA time'),
    'Oracle 23ai Vector Store':  (layer3_ml,        'RAG over past incidents enables intelligent diagnosis'),
    'OCI Data Science':          (layer3_ml,        'Custom models needed for domain-specific anomaly detection'),
    'Object Storage':            (layer2_ingestion, 'Needed for ML training data and model artefacts'),
    'ITSM / ServiceNow':         (layer4_automation,'Auto-ticket creation and closure completes the SRE feedback loop'),
}

score = 100
issues = []
warnings_list = []

for service, (layer_dict, reason) in critical.items():
    if not layer_dict.get(service, False):
        score -= 15
        issues.append((service, reason))

for service, (layer_dict, reason) in recommended.items():
    if not layer_dict.get(service, False):
        score -= 5
        warnings_list.append((service, reason))

score = max(score, 0)

print('ARCHITECTURE EVALUATION')
print('=' * 55)
print(f'  Score: {score}/100')

if score >= 90:
    grade = '✅  Production-ready AIOps architecture'
elif score >= 70:
    grade = '🟡  Good foundation — address warnings before production'
elif score >= 50:
    grade = '🟠  Critical gaps — will not achieve target MTTR'
else:
    grade = '🔴  Incomplete — pipeline cannot function end-to-end'
print(f'  Assessment: {grade}\n')

if issues:
    print('  🔴  CRITICAL GAPS (fix before deployment):')
    for svc, reason in issues:
        print(f'    • {svc}')
        print(f'      → {reason}')

if warnings_list:
    print('\n  🟡  RECOMMENDED ADDITIONS:')
    for svc, reason in warnings_list:
        print(f'    • {svc}')
        print(f'      → {reason}')

if not issues and not warnings_list:
    print('  ✅  No gaps found. Architecture covers all critical and recommended components.')

print('\n' + '=' * 55)
print('  PROJECTED OUTCOMES with this architecture:')

# Estimate impact based on which layers are complete
ml_complete    = all(layer3_ml.get(s, False)    for s in ['OCI Anomaly Detection Service', 'Correlation Engine'])
auto_complete  = all(layer4_automation.get(s, False) for s in ['OCI DevOps Pipelines', 'OCI Functions'])
genai_enabled  = layer3_ml.get('OCI GenAI Service', False)

alert_reduction = 40
mttr_reduction  = 30
if ml_complete:   alert_reduction += 30; mttr_reduction += 30
if auto_complete: mttr_reduction  += 25
if genai_enabled: mttr_reduction  += 10; alert_reduction += 5

current_alerts = 200
current_mttr   = 120  # minutes
new_alerts = int(current_alerts * (1 - alert_reduction/100))
new_mttr   = int(current_mttr   * (1 - mttr_reduction/100))

print(f'  Current alerts/day: {current_alerts}  →  Projected: {new_alerts}  ({alert_reduction}% reduction)')
print(f'  Current MTTR:       {current_mttr} min  →  Projected: {new_mttr} min  ({mttr_reduction}% reduction)')
target_met = '✅ Target met' if new_mttr <= 15 else f'⚠️  Target is 15 min — add more automation'
print(f'  MTTR target (15 min): {target_met}')

---
## Section 6 — Architecture Decision Record

An Architecture Decision Record (ADR) is a short document that explains **what you chose, why, and what you ruled out**.

Fill in your team/organisation details below and run the cell to generate your ADR.
This is a real deliverable you can take back to your team.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  FILL IN YOUR DETAILS
# ─────────────────────────────────────────────────────────────────────────────
organisation    = "Your Organisation Name"
platform        = "E-Commerce Platform on OCI"
author          = "Your Name"
date            = datetime.today().strftime('%d %B %Y')
primary_driver  = "Reduce MTTR from 2 hours to under 15 minutes"
secondary_driver= "Reduce alert volume from 200/day to under 40/day"
# ─────────────────────────────────────────────────────────────────────────────

selected_services = {
    layer_name: [s for s, v in layer.items() if v]
    for layer_name, layer in all_layers
}
excluded_services = [
    s for _, layer in all_layers for s, v in layer.items() if not v
]

adr = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║              ARCHITECTURE DECISION RECORD — OCI AIOps Platform              ║
╠══════════════════════════════════════════════════════════════════════════════╣
  Organisation : {organisation}
  Platform     : {platform}
  Author       : {author}
  Date         : {date}
  Status       : PROPOSED

──────────────────────────────────────────────────────────────────────────────
  CONTEXT
──────────────────────────────────────────────────────────────────────────────
  Primary driver  : {primary_driver}
  Secondary driver: {secondary_driver}
  Architecture    : Four-layer AIOps platform on OCI
                    Data Sources → Ingestion & Storage → ML & Analytics
                    → Automation & Feedback

──────────────────────────────────────────────────────────────────────────────
  DECISION — SELECTED OCI SERVICES
──────────────────────────────────────────────────────────────────────────────"""

for layer_name, services in selected_services.items():
    adr += f"\n  {layer_name}:"
    for s in services:
        adr += f"\n    • {s}"

adr += f"""

──────────────────────────────────────────────────────────────────────────────
  SERVICES EXCLUDED AND RATIONALE
──────────────────────────────────────────────────────────────────────────────"""
for s in excluded_services:
    adr += f"\n  • {s} — deferred to Phase 2 or not applicable to this scenario"

adr += f"""

──────────────────────────────────────────────────────────────────────────────
  CONSEQUENCES
──────────────────────────────────────────────────────────────────────────────
  Projected alert reduction : {alert_reduction}%   ({current_alerts}/day → {new_alerts}/day)
  Projected MTTR reduction  : {mttr_reduction}%   ({current_mttr} min  → {new_mttr} min)
  Architecture score        : {score}/100
  Assessment                : {grade}

  Risk: Model performance depends on volume of labelled incidents.
        Feedback loop must be active from day one to accumulate training data.
  Mitigation: Seed the vector store with historical incident data before go-live.

──────────────────────────────────────────────────────────────────────────────
  NEXT STEPS
──────────────────────────────────────────────────────────────────────────────
  1. Deploy OCI Logging + Monitoring + Service Connector Hub (Data foundation)
  2. Configure OCI Anomaly Detection Service on key metrics
  3. Build first self-healing pipeline in OCI DevOps
  4. Enable feedback loop to Object Storage from day one
  5. Add OCI GenAI Service + Oracle 23ai for intelligent RCA in Phase 2

╚══════════════════════════════════════════════════════════════════════════════╝
"""
print(adr)

# Save to file
with open('lab2_architecture_decision_record.txt', 'w') as f:
    f.write(adr)
print('ADR saved: lab2_architecture_decision_record.txt')

---
## Section 7 — Architecture Comparison: MVP vs Full AIOps

Most organisations cannot build the full architecture on day one.
This section shows what a **phased approach** looks like — and why the order of layers matters.

In [ ]:
phases = {
    'Phase 1\nData Foundation\n(Weeks 1–4)': {
        'services': ['OCI Logging', 'OCI Monitoring', 'Logging Analytics', 'Service Connector Hub'],
        'alert_reduction': 20, 'mttr_reduction': 15,
        'description': 'Centralise all telemetry. Eliminate manual log hunting.',
        'color': OCI_BLUE
    },
    'Phase 2\nML Detection\n(Weeks 5–10)': {
        'services': ['OCI Streaming', 'OCI Anomaly Detection', 'Correlation Engine', 'Object Storage'],
        'alert_reduction': 60, 'mttr_reduction': 40,
        'description': 'Add anomaly detection and event correlation. Kill alert storms.',
        'color': OCI_PURPLE
    },
    'Phase 3\nAutomation\n(Weeks 11–16)': {
        'services': ['OCI DevOps Pipelines', 'OCI Functions', 'OCI Events & Alarms', 'ITSM Integration'],
        'alert_reduction': 75, 'mttr_reduction': 70,
        'description': 'Auto-remediate. Self-heal. Close tickets without human touch.',
        'color': OCI_GREEN
    },
    'Phase 4\nIntelligence\n(Weeks 17+)': {
        'services': ['OCI GenAI Service', 'Oracle 23ai Vector Store', 'OCI Data Science', 'Feedback Loop'],
        'alert_reduction': 85, 'mttr_reduction': 87,
        'description': 'LLM diagnosis. RAG over incidents. Continuous model improvement.',
        'color': OCI_RED
    },
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Phased AIOps Adoption — Cumulative Impact per Phase', fontsize=13, fontweight='bold')

phase_labels = [p.split('\n')[0] + '\n' + p.split('\n')[1] for p in phases]
alert_vals   = [v['alert_reduction'] for v in phases.values()]
mttr_vals    = [v['mttr_reduction']  for v in phases.values()]
colors       = [v['color'] for v in phases.values()]

x = np.arange(len(phases))
for ax, vals, ylabel, title, baseline in [
    (axes[0], alert_vals, 'Cumulative alert reduction (%)', 'Alert Volume Reduction', 200),
    (axes[1], mttr_vals,  'Cumulative MTTR reduction (%)',  'MTTR Reduction',         120)
]:
    bars = ax.bar(x, vals, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5, width=0.55)
    ax.set_xticks(x)
    ax.set_xticklabels(phase_labels, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0, 100)
    ax.axhline(80, color='gray', linewidth=1, linestyle='--', alpha=0.5, label='Target 80%')
    ax.legend(fontsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f'{val}%', ha='center', fontsize=10, fontweight='bold', color=bar.get_facecolor())

plt.tight_layout()
plt.savefig('lab2_phased_adoption.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figure saved: lab2_phased_adoption.png')

print('\nKey insight: You cannot skip Phase 1 and 2 to get to Phase 4.')
print('The ML layer needs clean, centralised, streaming data to function.')
print('Automation needs ML outputs. Intelligence needs automation outcomes.')
print('The layers are a dependency chain — not a menu.')

---
## ✅ Lab Complete

### What you did in this lab

| Section | Activity |
|---|---|
| Section 2 | Studied the full OCI AIOps reference architecture — 4 layers, 20 services, live component diagram |
| Section 3 | Traced incident INC-2041 through all 4 layers — raw pod crash to auto-resolved in 30 seconds |
| Section 4 | Saw how the feedback loop improves MTTR and reduces false positives across 10 incidents |
| Section 5 | Designed your own architecture — selected OCI services per layer, got an automated evaluation |
| Section 6 | Generated an Architecture Decision Record to take back to your team |
| Section 7 | Mapped a phased adoption roadmap — Phase 1 to 4, with projected impact at each phase |

### Files generated
- `lab2_oci_architecture.png` — OCI AIOps reference architecture diagram
- `lab2_pipeline_timing.png` — Incident pipeline timing chart
- `lab2_feedback_loop.png` — Feedback loop improvement over time
- `lab2_phased_adoption.png` — Phased adoption roadmap
- `lab2_architecture_decision_record.txt` — Your Architecture Decision Record

---
**Next: Session 3 — Introduction to AI, ML and Generative AI**